* Modelo com constraint e normalização de price por sku 
* features de lag recursivas
* treinado separadamente para cada combinação sku-channel
* fazemos um teste para cada comparação de usar o método pooling ou agregado a nivel sku-channel para cada combinação (afim de implementar uma metodologia hibrida)

In [ ]:
"""
RetailCo Demand Model — Diagnostic Retrain
============================================
Business Case: Demand Model Optimization at RetailCo

Purpose
-------
1. Load the raw data (Table 1 - External Variables, Table 2 - Sell In).
2. Engineer features exactly as described in Table 4 (Feature Engineering Rules).
3. Train the BASELINE model using the current parameters (Table 3).
4. Train an IMPROVED model that targets the two root causes identified in the
   diagnostic phase:
     (a) Overfitting from excessive model capacity + zero regularization on a
         small weekly sample (156 weeks / SKU-channel).
     (b) Wrong-signed price elasticity (positive SHAP contribution for price),
         fixed with a native monotonic constraint on Price_per_kg_USD.
5. Evaluate both models with Accuracy = 1 - MAPE, per channel and overall.
6. Validate the Problem 2 fix with real SHAP values (sign consistency for
   price, and Promotion vs Advertising attribution share).
7. Root-cause follow-up: since price sign consistency alone doesn't fully
   resolve after the monotonic constraint, test a normalized Price_Index
   (price relative to each SKU's own average) to remove the cross-product
   price-scale confound created by pooling all 5 products into one model
   per channel.
8. Model bake-off: compare LightGBM (pooled and SKU-level), XGBoost
   (SKU-level), and a GAM (SKU-level, with explicit smooth trend and
   seasonality terms) to see which family/architecture combination wins
   per SKU-channel, using the RECURSIVE (multi-step) evaluation, which is
   the one representative of the real 78-week forecast.

Requirements
------------
pip install lightgbm xgboost pygam shap pandas numpy scikit-learn matplotlib openpyxl

NOTE on pygam: it's a smaller, less actively maintained package than
lightgbm/xgboost. If `pip install pygam` fails, try `pip install pygam --no-build-isolation`
or check https://pygam.readthedocs.io for platform-specific install notes.

Usage
-----
1. Edit DATA_PATH below (in the CONFIG section) to point to your Excel file.
2. Run either:
   - In a terminal:        python retrain_demand_model.py
   - In Jupyter/Colab:     run all cells, then call main() in the last cell
"""

# ---------------------------------------------------------------------------
# 1. LIBRARIES
# ---------------------------------------------------------------------------

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
from pygam import LinearGAM, s, l, f

warnings.filterwarnings("ignore")


# ---------------------------------------------------------------------------
# 2. CONFIG
# ---------------------------------------------------------------------------

# >>> EDIT THESE TWO LINES <<<
# Path to the Business Case Data Set Excel file on your machine.
DATA_PATH = "Business_Case_Data_Set.xlsx"
# Folder where results (CSV + plots) will be saved. Created automatically.
OUTDIR = "output"

FEATURE_COLS = [
    "Price_per_kg_USD",
    "Numeric_Distribution",
    "Weighted_Distribution",
    "Advertising_Investment_USD",
    "Promotion_Investment_USD",
    "Promo_Flag",
    "Lag_1w",
    "Lag_4w",
    "Rolling_Mean_4w",
    "Rolling_Std_4w",
    "Price_Change_Pct",
    "Month_Sin",
    "Month_Cos",
    "Holiday_Flag",
    "Interaction_Price_Promo",
]

TARGET_COL = "Sell_In_Tons"

# Feature set for the price-normalization variant (used later by
# add_price_index / IMPROVED_V3_PARAMS): identical to FEATURE_COLS, but
# Price_per_kg_USD -> Price_Index and Interaction_Price_Promo ->
# Interaction_PriceIndex_Promo.
FEATURE_COLS_NORM = [
    {"Price_per_kg_USD": "Price_Index", "Interaction_Price_Promo": "Interaction_PriceIndex_Promo"}.get(c, c)
    for c in FEATURE_COLS
]

HOLDOUT_QUANTILE = 0.85  # last ~15% of weeks per SKU-channel used as test set

CATEGORY_MAP = {
    "Natural Juice 1L": "Beverages",
    "Flavored Water 500ml": "Beverages",
    "Energy Drink 350ml": "Beverages",
    "Whole Grain Crackers 200g": "Snacks",
    "Cereal Bar 50g": "Snacks",
}

# Params exactly as reported in Table 3 - Model Parameters
BASELINE_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=12,
    num_leaves=256,
    min_child_samples=5,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.0,
    reg_lambda=0.0,
    min_split_gain=0.0,
    boosting_type="gbdt",
    objective="regression",
    metric="mape",
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

# Regularized + monotonic-constrained candidate.
# monotone_constraints: -1 forces Price_per_kg_USD to have a non-positive
# effect on predicted volume (correct economic sign). All other features
# are left unconstrained (0).
IMPROVED_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=6,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=1.0,
    reg_lambda=1.0,
    min_split_gain=0.0,
    boosting_type="gbdt",
    objective="regression",
    metric="mape",
    random_state=42,
    n_jobs=-1,
    verbose=-1,
    monotone_constraints=[
        -1 if col == "Price_per_kg_USD" else 0 for col in FEATURE_COLS
    ],
    monotone_constraints_method="advanced",
)

# Diagnostic variant: same as IMPROVED_PARAMS, but ALSO constrains
# Interaction_Price_Promo (= Price_per_kg_USD * Promo_Flag) to be
# non-increasing in price. Tests the hypothesis that the positive price
# SHAP signal is "leaking" through the unconstrained interaction term.
IMPROVED_V2_PARAMS = dict(
    IMPROVED_PARAMS,
    monotone_constraints=[
        -1 if col in ("Price_per_kg_USD", "Interaction_Price_Promo") else 0
        for col in FEATURE_COLS
    ],
)

# V3: same regularization as IMPROVED_PARAMS, but built for FEATURE_COLS_NORM
# (raw price replaced with the SKU-relative Price_Index). Constrains
# Price_Index to be non-increasing, same economic logic as before.
IMPROVED_V3_PARAMS = dict(
    IMPROVED_PARAMS,
    monotone_constraints=[
        -1 if col == "Price_Index" else 0 for col in FEATURE_COLS_NORM
    ],
)

# SKU-level variant: one model PER Product x Channel combo, instead of
# pooling all 5 products into one model per channel. Each model now sees
# only ~130 training weeks (vs ~650 pooled), so the tree is deliberately
# simpler than IMPROVED_PARAMS to avoid overfitting on less data. No
# monotonic constraint here -- this test isolates the effect of NOT
# pooling on raw accuracy, independent of the Problem 2 fix.
SKU_LEVEL_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=4,
    num_leaves=15,
    min_child_samples=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.0,
    min_split_gain=0.0,
    boosting_type="gbdt",
    objective="regression",
    metric="mape",
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

# --- Model bake-off: XGBoost and GAM, both trained SKU-level (same
# architecture as the LightGBM SKU-level winner), to compare model
# FAMILIES under a fixed, already-validated architecture. ---

# XGBoost mirrors SKU_LEVEL_PARAMS as closely as XGBoost's own parameter
# names allow. monotone_constraints uses XGBoost's native tuple format
# (one entry per feature in FEATURE_COLS, in order): -1 = non-increasing,
# 0 = unconstrained. Same economic logic as the LightGBM constraint.
XGB_SKU_PARAMS = dict(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=4,
    min_child_weight=5,       # analogous to LightGBM's min_child_samples
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.0,
    gamma=0.0,                 # analogous to LightGBM's min_split_gain
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
    monotone_constraints=tuple(-1 if c == "Price_per_kg_USD" else 0 for c in FEATURE_COLS),
)

# GAM feature set: swaps the tree-style lag/rolling features for an
# explicit smooth TREND term (Week_Index) -- this is the whole point of
# testing a GAM. Trend and seasonality (Month_Sin/Cos) get their own
# additive smooth terms instead of being inferred indirectly through
# lag features and tree splits.
FEATURE_COLS_GAM = [
    "Week_Index",
    "Month_Sin",
    "Month_Cos",
    "Price_per_kg_USD",
    "Numeric_Distribution",
    "Weighted_Distribution",
    "Advertising_Investment_USD",
    "Promotion_Investment_USD",
    "Promo_Flag",
    "Lag_1w",
    "Price_Change_Pct",
    "Interaction_Price_Promo",
]
GAM_N_SPLINES = 8   # kept low deliberately -- ~130 training rows per SKU-channel
GAM_LAM = 0.6       # regularization strength (higher = smoother/less overfit)


# ---------------------------------------------------------------------------
# 3. LOAD DATA
# ---------------------------------------------------------------------------

def load_data(path: str) -> pd.DataFrame:
    """
    Loads Table 1 (External Variables) and Table 2 (Sell In) from the
    Business Case Data Set and merges them on Week/Product/Channel.
    Returns the raw merged dataframe, BEFORE any feature engineering.
    """
    xl = pd.ExcelFile(path)
    ext = xl.parse("Table 1 - External Variables")
    actual = xl.parse("Table 2 - Sell In")

    df = ext.merge(actual, on=["Week", "Product", "Channel"], how="inner")
    df["Week"] = pd.to_datetime(df["Week"])
    df["Category"] = df["Product"].map(CATEGORY_MAP)
    df = df.sort_values(["Product", "Channel", "Week"]).reset_index(drop=True)

    # Trend term for the GAM: position (0, 1, 2, ...) of each week within
    # its own Product x Channel series, computed on the FULL raw series
    # (before engineer_features() drops the early rows with no lag
    # history). This keeps it numerically consistent with the recursive
    # backtest, where the history buffer also starts from the full series.
    df["Week_Index"] = df.groupby(["Product", "Channel"]).cumcount()

    return df


# ---------------------------------------------------------------------------
# 4. FEATURE ENGINEERING (mirrors Table 4 - Feature Engineering Rules)
# ---------------------------------------------------------------------------

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Applies the 10 feature engineering rules from Table 4 to the raw
    merged dataframe. Returns a dataframe with all FEATURE_COLS populated,
    with rows containing NaN (e.g. the first weeks of each SKU-channel,
    which have no lag history) dropped.
    """
    df = df.copy()

    # FE_06 Promo_Flag: 1 if Promotion_Type != None, else 0
    df["Promo_Flag"] = df["Promotion_Type"].notna().astype(int)

    g_target = df.groupby(["Product", "Channel"])[TARGET_COL]

    # FE_01 / FE_02: lags of Sell-In volume
    df["Lag_1w"] = g_target.shift(1)
    df["Lag_4w"] = g_target.shift(4)

    # FE_03 / FE_04: rolling mean/std, computed on the SHIFTED series
    # so the current week's own volume is never used to predict itself
    shifted = g_target.shift(1)
    grouped_shifted = shifted.groupby([df["Product"], df["Channel"]])
    df["Rolling_Mean_4w"] = grouped_shifted.rolling(4).mean().reset_index(
        level=[0, 1], drop=True
    )
    df["Rolling_Std_4w"] = grouped_shifted.rolling(4).std().reset_index(
        level=[0, 1], drop=True
    )

    # FE_05: week-over-week % change in price
    df["Price_Change_Pct"] = df.groupby(["Product", "Channel"])[
        "Price_per_kg_USD"
    ].pct_change()

    # FE_07 / FE_08: cyclical month encoding
    month = df["Week"].dt.month
    df["Month_Sin"] = np.sin(2 * np.pi * month / 12)
    df["Month_Cos"] = np.cos(2 * np.pi * month / 12)

    # FE_09: Holiday_Flag
    # ASSUMPTION: no explicit holiday calendar table was provided in the
    # dataset, so this defaults to 0. If you have a real calendar, merge a
    # Holiday_Flag column keyed by Week BEFORE calling this function, and
    # remove the line below.
    df["Holiday_Flag"] = 0

    # FE_10: interaction term between price and promotion
    df["Interaction_Price_Promo"] = df["Price_per_kg_USD"] * df["Promo_Flag"]

    df = df.dropna(subset=FEATURE_COLS + [TARGET_COL]).reset_index(drop=True)
    return df


def time_split(df: pd.DataFrame):
    """
    Time-based train/test split, computed independently per SKU-channel so
    every series contributes its most recent ~15% of weeks to the test set.
    This avoids leakage (no future data used to predict the past).
    """
    cutoff = df.groupby(["Product", "Channel"])["Week"].transform(
        lambda x: x.quantile(HOLDOUT_QUANTILE, interpolation="nearest")
    )
    train = df[df["Week"] <= cutoff].copy()
    test = df[df["Week"] > cutoff].copy()
    return train, test


def add_price_index(train: pd.DataFrame, test: pd.DataFrame):
    """
    Root-cause fix: models are trained pooling ALL 5 products within a
    channel, but products sit on very different price scales (e.g.
    Flavored Water ~$1.80/kg vs Cereal Bar ~$12/kg). SHAP explains a
    feature relative to the POPULATION average, so a structurally cheap
    product like Natural Juice can show a "wrong-signed" price SHAP even
    when the model's actual price-volume relationship is correct for
    that SKU (confirmed via partial dependence).

    Fix: replace the raw price with a self-referential index --
        Price_Index = Price_per_kg_USD / (product's own average price)
    so 1.0 always means "this SKU's own typical price", regardless of
    the SKU's absolute price level. The per-product average is computed
    from the TRAINING set only, then applied to both train and test, to
    avoid leakage.
    """
    product_mean_price = train.groupby("Product")["Price_per_kg_USD"].mean()

    train = train.copy()
    test = test.copy()
    train["Price_Index"] = train["Price_per_kg_USD"] / train["Product"].map(product_mean_price)
    test["Price_Index"] = test["Price_per_kg_USD"] / test["Product"].map(product_mean_price)

    train["Interaction_PriceIndex_Promo"] = train["Price_Index"] * train["Promo_Flag"]
    test["Interaction_PriceIndex_Promo"] = test["Price_Index"] * test["Promo_Flag"]

    return train, test


# ---------------------------------------------------------------------------
# 5. MODEL TRAINING
# ---------------------------------------------------------------------------

class GAMModelWrapper:
    """
    Wraps pygam.LinearGAM with a fit(X, y) / predict(X) interface
    compatible with every other function in this script (train_model,
    evaluate_*, recursive_backtest_*, price_partial_dependence all just
    call .fit(X, y) / .predict(X) on a DataFrame).

    Unlike the tree models, the GAM needs an explicit smooth term for
    trend (Week_Index) and models seasonality/economic drivers as
    separate additive terms -- trend and seasonality are captured
    directly and smoothly, instead of being inferred indirectly through
    lag features and tree splits. Price gets a monotonic-decreasing
    smooth term, mirroring the same economic constraint used on the
    LightGBM/XGBoost models.

    NOTE: pygam's exact keyword arguments (constraints=, n_splines=,
    lam=) match the stable public API as of pygam 0.8.x. If you hit a
    TypeError on these kwargs, check your installed pygam version's docs.
    """

    def __init__(self, feature_cols: list, n_splines: int = 8, lam: float = 0.6):
        self.feature_cols = feature_cols
        self.n_splines = n_splines
        self.lam = lam
        self.model = None

    def _build_terms(self):
        terms = None
        for i, col in enumerate(self.feature_cols):
            if col == "Week_Index":
                term = s(i, n_splines=self.n_splines, lam=self.lam)
            elif col == "Price_per_kg_USD":
                term = s(i, n_splines=self.n_splines, lam=self.lam, constraints="monotonic_dec")
            elif col in ("Month_Sin", "Month_Cos"):
                term = s(i, n_splines=self.n_splines, lam=self.lam)
            elif col == "Promo_Flag":
                term = f(i)
            else:
                term = l(i)
            terms = term if terms is None else terms + term
        return terms

    def fit(self, X: pd.DataFrame, y: pd.Series):
        terms = self._build_terms()
        self.model = LinearGAM(terms).fit(X[self.feature_cols].values, y.values)
        return self

    def predict(self, X: pd.DataFrame):
        return self.model.predict(X[self.feature_cols].values)


def train_model(X: pd.DataFrame, y: pd.Series, params: dict, model_class=None):
    """
    Fits a single regressor with the given parameter set. Defaults to
    LightGBM; pass model_class=xgb.XGBRegressor (or any sklearn-API-
    compatible class) to train a different family with this same
    function -- everything downstream (evaluation, recursive backtest,
    SHAP for tree models) keeps working unchanged.
    """
    model_class = model_class or lgb.LGBMRegressor
    model = model_class(**params)
    model.fit(X, y)
    return model


def train_all_channels(train: pd.DataFrame, params: dict, feature_cols: list = None, model_class=None) -> dict:
    """
    Trains one model per channel (mirrors RetailCo's current slicing
    approach: one model per channel, sharing the same parameter set).
    Returns {channel: fitted_model}.
    """
    feature_cols = feature_cols or FEATURE_COLS
    models = {}
    for channel in train["Channel"].unique():
        sub = train[train["Channel"] == channel]
        X, y = sub[feature_cols], sub[TARGET_COL]
        models[channel] = train_model(X, y, params, model_class=model_class)
    return models


def train_all_sku_channels(train: pd.DataFrame, params: dict, feature_cols: list = None, model_class=None) -> dict:
    """
    Trains one model PER Product x Channel combination -- no pooling across
    products. Returns {(product, channel): fitted_model}.

    Trade-off vs train_all_channels(): each model sees far less data
    (~130 weeks vs ~650 pooled), but by construction it can never confuse
    price scale across products (no Price_Index normalization needed --
    the model only ever sees ONE product's price range).
    """
    feature_cols = feature_cols or FEATURE_COLS
    models = {}
    for (product, channel), sub in train.groupby(["Product", "Channel"]):
        if len(sub) < 15:
            continue  # not enough history to train a meaningful model
        X, y = sub[feature_cols], sub[TARGET_COL]
        models[(product, channel)] = train_model(X, y, params, model_class=model_class)
    return models


def train_gam_sku_channels(train: pd.DataFrame, feature_cols: list = None, n_splines: int = None, lam: float = None) -> dict:
    """
    Trains one GAM per Product x Channel combination. Kept as its own
    function (rather than routing through train_model/train_all_sku_channels)
    because GAMModelWrapper's constructor takes structurally different
    arguments (feature_cols, n_splines, lam) than a flat hyperparameter
    dict shared with the tree models.
    """
    feature_cols = feature_cols or FEATURE_COLS_GAM
    n_splines = n_splines or GAM_N_SPLINES
    lam = lam if lam is not None else GAM_LAM
    models = {}
    for (product, channel), sub in train.groupby(["Product", "Channel"]):
        if len(sub) < 15:
            continue
        X, y = sub[feature_cols], sub[TARGET_COL]
        wrapper = GAMModelWrapper(feature_cols, n_splines=n_splines, lam=lam)
        wrapper.fit(X, y)
        models[(product, channel)] = wrapper
    return models


# ---------------------------------------------------------------------------
# 6. MODEL EVALUATION — Accuracy = 1 - MAPE
# ---------------------------------------------------------------------------

def evaluate_model(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """
    Computes MAPE and Accuracy (1 - MAPE), matching the metric definition
    used throughout the business case (Table 3: metric = 'mape';
    Problem Statement: Model Accuracy = 1 - MAPE).
    """
    mape = float(np.mean(np.abs(y_true - y_pred) / y_true))
    return {"MAPE": mape, "Accuracy": 1 - mape}


def evaluate_all_channels(models: dict, test: pd.DataFrame, feature_cols: list = None) -> pd.DataFrame:
    """
    Evaluates a set of per-channel models against the test set and returns
    a per-channel + overall (volume-weighted) accuracy table.
    """
    feature_cols = feature_cols or FEATURE_COLS
    rows = []
    for channel, model in models.items():
        sub = test[test["Channel"] == channel]
        if len(sub) == 0:
            continue
        X, y_true = sub[feature_cols], sub[TARGET_COL].values
        y_pred = model.predict(X)
        metrics = evaluate_model(y_true, y_pred)
        rows.append({"Channel": channel, "n_test": len(sub), **metrics})

    result = pd.DataFrame(rows)
    overall_mape = np.average(result["MAPE"], weights=result["n_test"])
    overall_row = pd.DataFrame(
        [{
            "Channel": "OVERALL (weighted)",
            "n_test": result["n_test"].sum(),
            "MAPE": overall_mape,
            "Accuracy": 1 - overall_mape,
        }]
    )
    return pd.concat([result, overall_row], ignore_index=True)


def compare_baseline_vs_improved(
    baseline_eval: pd.DataFrame, improved_eval: pd.DataFrame
) -> pd.DataFrame:
    """Merges the two evaluation tables side by side for reporting."""
    merged = baseline_eval.merge(
        improved_eval, on="Channel", suffixes=("_baseline", "_improved")
    )
    merged["Accuracy_delta_pp"] = (
        merged["Accuracy_improved"] - merged["Accuracy_baseline"]
    ) * 100
    return merged[
        [
            "Channel",
            "n_test_baseline",
            "MAPE_baseline",
            "Accuracy_baseline",
            "MAPE_improved",
            "Accuracy_improved",
            "Accuracy_delta_pp",
        ]
    ].rename(columns={"n_test_baseline": "n_test"})


def evaluate_all_sku_channels(models: dict, test: pd.DataFrame, feature_cols: list = None) -> pd.DataFrame:
    """
    Evaluates SKU-channel-level models (keyed by (product, channel)).
    Returns a per-SKU-channel + overall (volume-weighted) accuracy table.
    """
    feature_cols = feature_cols or FEATURE_COLS
    rows = []
    for (product, channel), model in models.items():
        sub = test[(test["Product"] == product) & (test["Channel"] == channel)]
        if len(sub) == 0:
            continue
        X, y_true = sub[feature_cols], sub[TARGET_COL].values
        y_pred = model.predict(X)
        metrics = evaluate_model(y_true, y_pred)
        rows.append({"Product": product, "Channel": channel, "n_test": len(sub), **metrics})

    result = pd.DataFrame(rows)
    overall_mape = np.average(result["MAPE"], weights=result["n_test"])
    overall_row = pd.DataFrame(
        [{
            "Product": "ALL",
            "Channel": "OVERALL (weighted)",
            "n_test": result["n_test"].sum(),
            "MAPE": overall_mape,
            "Accuracy": 1 - overall_mape,
        }]
    )
    return pd.concat([result, overall_row], ignore_index=True)


def aggregate_sku_eval_by_channel(sku_eval: pd.DataFrame) -> pd.DataFrame:
    """
    Rolls the per-SKU-channel evaluation up to the channel level, so it can
    be compared apples-to-apples against the pooled model's evaluate_all_channels()
    output (same test weeks, same weighting).
    """
    df = sku_eval[sku_eval["Product"] != "ALL"]
    rows = []
    for channel, sub in df.groupby("Channel"):
        mape = np.average(sub["MAPE"], weights=sub["n_test"])
        rows.append({"Channel": channel, "n_test": int(sub["n_test"].sum()), "MAPE": mape, "Accuracy": 1 - mape})
    result = pd.DataFrame(rows)
    overall_mape = np.average(result["MAPE"], weights=result["n_test"])
    overall_row = pd.DataFrame(
        [{"Channel": "OVERALL (weighted)", "n_test": int(result["n_test"].sum()),
          "MAPE": overall_mape, "Accuracy": 1 - overall_mape}]
    )
    return pd.concat([result, overall_row], ignore_index=True)


def evaluate_pooled_by_sku(models: dict, test: pd.DataFrame, feature_cols: list = None) -> pd.DataFrame:
    """
    Evaluates the POOLED (per-channel) models, but broken down by individual
    Product -- i.e. "how good is the pooled Supermarkets model specifically
    for Natural Juice rows?". This is what makes the pooled vs SKU-level
    comparison apples-to-apples: same test rows, same weeks, only the
    model that produced the prediction differs.
    """
    feature_cols = feature_cols or FEATURE_COLS
    rows = []
    for channel, model in models.items():
        sub_channel = test[test["Channel"] == channel]
        for product in sub_channel["Product"].unique():
            sub = sub_channel[sub_channel["Product"] == product]
            if len(sub) == 0:
                continue
            X, y_true = sub[feature_cols], sub[TARGET_COL].values
            y_pred = model.predict(X)
            metrics = evaluate_model(y_true, y_pred)
            rows.append({"Product": product, "Channel": channel, "n_test": len(sub), **metrics})
    return pd.DataFrame(rows)


def compare_pooled_vs_sku_level(
    pooled_by_sku: pd.DataFrame, sku_level_eval: pd.DataFrame, min_accuracy: float = 0.0
) -> pd.DataFrame:
    """
    Builds the routing table: for each Product x Channel, which approach
    wins on accuracy, with a safety rule -- if the SKU-level model's
    accuracy falls below `min_accuracy` (default: 0, i.e. worse than just
    predicting the average), it's flagged as unstable and Pooled is
    recommended regardless of the raw comparison, since a model that
    underperforms a naive average shouldn't be trusted in production even
    if it happens to win narrowly on a single small test set.
    """
    sku_level = sku_level_eval[sku_level_eval["Product"] != "ALL"]
    merged = pooled_by_sku.merge(
        sku_level, on=["Product", "Channel"], suffixes=("_pooled", "_sku_level")
    )
    merged["Accuracy_delta_pp"] = (
        merged["Accuracy_sku_level"] - merged["Accuracy_pooled"]
    ) * 100

    def recommend(row):
        if row["Accuracy_sku_level"] < min_accuracy:
            return "Pooled (SKU-level unstable)"
        return "SKU-level" if row["Accuracy_sku_level"] > row["Accuracy_pooled"] else "Pooled"

    merged["Recommendation"] = merged.apply(recommend, axis=1)
    cols = [
        "Product", "Channel", "n_test_pooled",
        "Accuracy_pooled", "Accuracy_sku_level", "Accuracy_delta_pp", "Recommendation",
    ]
    return merged[cols].rename(columns={"n_test_pooled": "n_test"}).sort_values(
        "Accuracy_delta_pp", ascending=False
    ).reset_index(drop=True)


def compare_pooled_vs_sku_level_recursive(
    recursive_pooled: pd.DataFrame, recursive_sku_level: pd.DataFrame, min_accuracy: float = 0.0
) -> pd.DataFrame:
    """
    Same routing logic as compare_pooled_vs_sku_level(), but built from
    RECURSIVE (multi-step) backtests for BOTH sides -- this is the version
    that should actually drive the pooled vs SKU-level decision, since it
    reflects what happens forecasting 78 weeks with no real data arriving
    in between (the real deployment scenario), instead of mixing a
    one-step-ahead evaluation for one side with a recursive one for the
    other.

    `recursive_pooled` = output of recursive_backtest_all(pooled_models, raw)
    `recursive_sku_level` = output of recursive_backtest_all_sku_level(sku_level_models, raw)
    """
    pooled_acc = recursive_accuracy_by_sku(recursive_pooled).rename(
        columns={"Accuracy": "Accuracy_pooled", "n": "n_test"}
    )
    sku_acc = recursive_accuracy_by_sku(recursive_sku_level).rename(
        columns={"Accuracy": "Accuracy_sku_level"}
    )
    merged = pooled_acc.merge(sku_acc[["Product", "Channel", "Accuracy_sku_level"]], on=["Product", "Channel"])
    merged["Accuracy_delta_pp"] = (merged["Accuracy_sku_level"] - merged["Accuracy_pooled"]) * 100

    def recommend(row):
        if row["Accuracy_sku_level"] < min_accuracy:
            return "Pooled (SKU-level unstable)"
        return "SKU-level" if row["Accuracy_sku_level"] > row["Accuracy_pooled"] else "Pooled"

    merged["Recommendation"] = merged.apply(recommend, axis=1)
    cols = ["Product", "Channel", "n_test", "Accuracy_pooled", "Accuracy_sku_level", "Accuracy_delta_pp", "Recommendation"]
    return merged[cols].sort_values("Accuracy_delta_pp", ascending=False).reset_index(drop=True)


def build_model_bakeoff_table(accuracy_tables: dict, min_accuracy: float = 0.0) -> pd.DataFrame:
    """
    Generalizes compare_pooled_vs_sku_level_recursive() to N candidates
    instead of just 2. Builds one row per Product x Channel, with each
    candidate's accuracy as its own column, plus the winner.

    accuracy_tables: {label: dataframe} where each dataframe has columns
    Product, Channel, Accuracy (e.g. the output of recursive_accuracy_by_sku()
    for each candidate model/architecture combination being compared).

    Safety rule: if even the BEST candidate's accuracy is below
    `min_accuracy` (default 0, i.e. worse than predicting the average),
    the recommendation is "NONE stable" instead of picking a technical
    winner that still shouldn't be trusted in production.
    """
    merged = None
    for label, acc_df in accuracy_tables.items():
        renamed = acc_df[["Product", "Channel", "Accuracy"]].rename(columns={"Accuracy": label})
        merged = renamed if merged is None else merged.merge(renamed, on=["Product", "Channel"])

    labels = list(accuracy_tables.keys())
    merged["Best_Model"] = merged[labels].idxmax(axis=1)
    merged["Best_Accuracy"] = merged[labels].max(axis=1)
    merged["Recommendation"] = np.where(
        merged["Best_Accuracy"] < min_accuracy,
        "NONE stable -- needs further investigation",
        merged["Best_Model"],
    )
    return merged.sort_values("Best_Accuracy", ascending=False).reset_index(drop=True)


# ---------------------------------------------------------------------------
# 7. SHAP VALIDATION — Problem 2 (interpretability / forecast decomposition)
# ---------------------------------------------------------------------------

def price_sign_consistency(
    models: dict = None,
    df: pd.DataFrame = None,
    product: str = "Natural Juice 1L",
    channel: str = "Supermarkets",
    feature_cols: list = None,
    price_col: str = "Price_per_kg_USD",
    model=None,
) -> dict:
    """
    % of observations where SHAP attributes a POSITIVE contribution to
    the price feature (i.e. the wrong economic sign) for a given model.
    Works for either the raw price (price_col="Price_per_kg_USD") or the
    normalized index (price_col="Price_Index"), matching feature_cols.

    Pass either `models` (a {channel: model} dict, as returned by
    train_all_channels) OR `model` directly (e.g. a single SKU-channel
    model from train_all_sku_channels).
    """
    feature_cols = feature_cols or FEATURE_COLS
    subset = df[(df["Product"] == product) & (df["Channel"] == channel)]
    X = subset[feature_cols]

    if model is None:
        model = models[channel]
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)
    price_idx = feature_cols.index(price_col)
    price_shap = shap_values[:, price_idx]

    return {
        "pct_positive_sign": float((price_shap > 0).mean() * 100),
        "mean_shap": float(price_shap.mean()),
    }


def promo_vs_advertising_share(
    models: dict,
    df: pd.DataFrame,
    channel: str = "E-commerce",
    category: str = "Beverages",
) -> dict:
    """
    % share of total |SHAP| attributed to Promotion-related features vs
    Advertising Investment, for a given channel/category.
    """
    subset = df[(df["Channel"] == channel) & (df["Category"] == category)]
    X = subset[FEATURE_COLS]

    model = models[channel]
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    abs_total = np.abs(shap_values).sum()
    promo_idx = [
        FEATURE_COLS.index(c)
        for c in ("Promotion_Investment_USD", "Promo_Flag", "Interaction_Price_Promo")
    ]
    adv_idx = FEATURE_COLS.index("Advertising_Investment_USD")

    return {
        "promo_share_pct": float(np.abs(shap_values[:, promo_idx]).sum() / abs_total * 100),
        "advertising_share_pct": float(np.abs(shap_values[:, adv_idx]).sum() / abs_total * 100),
    }


def price_partial_dependence(
    model,
    df: pd.DataFrame,
    product: str = "Natural Juice 1L",
    channel: str = "Supermarkets",
    n_points: int = 20,
    feature_cols: list = None,
    price_col: str = "Price_per_kg_USD",
) -> pd.DataFrame:
    """
    Direct proof that the monotonic constraint holds: sweeps the price
    feature across its observed range (holding every other feature fixed
    at each row's actual values) and returns the AVERAGE predicted volume
    at each price point. Unlike the SHAP sign check, this must be
    non-increasing by construction when the model was trained with a
    -1 monotonic constraint on that feature.
    """
    feature_cols = feature_cols or FEATURE_COLS
    subset = df[(df["Product"] == product) & (df["Channel"] == channel)]
    X = subset[feature_cols].copy()

    price_min, price_max = X[price_col].min(), X[price_col].max()
    grid = np.linspace(price_min, price_max, n_points)

    avg_preds = []
    for p in grid:
        X_sweep = X.copy()
        X_sweep[price_col] = p
        avg_preds.append(model.predict(X_sweep).mean())

    return pd.DataFrame({price_col: grid, "Avg_Predicted_Volume": avg_preds})


def plot_partial_dependence(
    pdp_baseline: pd.DataFrame, pdp_improved: pd.DataFrame, out_path: str,
    price_col: str = "Price_per_kg_USD", x_label: str = "Price per kg (USD)",
    title_suffix: str = "", label_baseline: str = "Baseline (atual)",
    label_improved: str = "Improved (monotonic)",
):
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.plot(
        pdp_baseline[price_col], pdp_baseline["Avg_Predicted_Volume"],
        marker="o", label=label_baseline, color="#d62728",
    )
    ax.plot(
        pdp_improved[price_col], pdp_improved["Avg_Predicted_Volume"],
        marker="o", label=label_improved, color="#2ca02c",
    )
    ax.set_xlabel(x_label)
    ax.set_ylabel("Avg predicted volume (tons)")
    ax.set_title(f"Partial dependence: Price vs Predicted Volume{title_suffix}\nNatural Juice / Supermarkets")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


# ---------------------------------------------------------------------------
# 7B. RECURSIVE (MULTI-STEP) BACKTEST
# ---------------------------------------------------------------------------
#
# The evaluation above (evaluate_all_channels) is a ONE-STEP-AHEAD backtest:
# lag/rolling features in the test set are built from the REAL Sell_In_Tons,
# simulating a weekly reforecast where fresh actuals are always available.
#
# That is NOT what will happen for the 78-week future forecast: there is no
# real Sell_In there, so Lag_1w / Lag_4w / Rolling_Mean_4w / Rolling_Std_4w
# must be built from the model's OWN previous predictions, recursively. This
# section replays the same holdout period, but recursively, to measure how
# fast accuracy degrades as the forecast horizon grows -- exactly the
# behavior we'll see in the real 78-week forecast.

def recursive_backtest_sku(
    model,
    raw_df: pd.DataFrame,
    product: str,
    channel: str,
    feature_cols: list = None,
) -> pd.DataFrame:
    """
    Runs a true multi-step-ahead backtest for a single Product/Channel:
    walks forward week by week through the holdout period, and instead of
    using the real Sell_In_Tons to build Lag_1w/Lag_4w/Rolling_* (like the
    one-step-ahead backtest does), feeds the MODEL'S OWN prior predictions
    back in -- exactly what happens when forecasting 78 weeks with no real
    data arriving in between.

    `raw_df` must be the raw merged dataframe from load_data() (BEFORE
    engineer_features), so it still has one row per Product/Channel/Week
    with the untouched external variables and the real Sell_In_Tons (used
    only for computing accuracy afterwards, never fed into the features).
    """
    feature_cols = feature_cols or FEATURE_COLS
    sub = (
        raw_df[(raw_df["Product"] == product) & (raw_df["Channel"] == channel)]
        .sort_values("Week")
        .reset_index(drop=True)
    )

    cutoff = sub["Week"].quantile(HOLDOUT_QUANTILE, interpolation="nearest")
    train_idx = sub.index[sub["Week"] <= cutoff].tolist()
    test_idx = sub.index[sub["Week"] > cutoff].tolist()
    if len(test_idx) < 2 or len(train_idx) < 4:
        return pd.DataFrame()

    # History buffer starts as the REAL Sell_In for every training week
    # (this is legitimate "burn-in" -- those are real past actuals, known
    # at forecast time). From here on, only predictions get appended.
    history = sub.loc[train_idx, TARGET_COL].tolist()

    rows = []
    for horizon, idx in enumerate(test_idx, start=1):
        row = sub.loc[idx]

        promo_flag = int(pd.notna(row["Promotion_Type"]))
        prev_price = sub.loc[idx - 1, "Price_per_kg_USD"] if (idx - 1) in sub.index else np.nan
        price_change_pct = (
            (row["Price_per_kg_USD"] - prev_price) / prev_price
            if pd.notna(prev_price) and prev_price != 0 else np.nan
        )
        month = row["Week"].month

        feat = {
            "Week_Index": len(history),
            "Price_per_kg_USD": row["Price_per_kg_USD"],
            "Numeric_Distribution": row["Numeric_Distribution"],
            "Weighted_Distribution": row["Weighted_Distribution"],
            "Advertising_Investment_USD": row["Advertising_Investment_USD"],
            "Promotion_Investment_USD": row["Promotion_Investment_USD"],
            "Promo_Flag": promo_flag,
            "Lag_1w": history[-1],
            "Lag_4w": history[-4] if len(history) >= 4 else history[0],
            "Rolling_Mean_4w": float(np.mean(history[-4:])),
            "Rolling_Std_4w": float(np.std(history[-4:], ddof=1)) if len(history[-4:]) > 1 else 0.0,
            "Price_Change_Pct": price_change_pct,
            "Month_Sin": np.sin(2 * np.pi * month / 12),
            "Month_Cos": np.cos(2 * np.pi * month / 12),
            "Holiday_Flag": 0,
            "Interaction_Price_Promo": row["Price_per_kg_USD"] * promo_flag,
        }
        X_row = pd.DataFrame([feat])[feature_cols]
        pred = float(model.predict(X_row)[0])

        rows.append({
            "Product": product,
            "Channel": channel,
            "Week": row["Week"],
            "Horizon": horizon,
            "Actual": row[TARGET_COL],
            "Predicted": pred,
        })

        # Feed the PREDICTION back -- never the real value -- simulating
        # no actuals arriving during the forecast window.
        history.append(pred)

    result = pd.DataFrame(rows)
    result["APE"] = (result["Actual"] - result["Predicted"]).abs() / result["Actual"]
    return result


def recursive_backtest_all(model_dict: dict, raw_df: pd.DataFrame, feature_cols: list = None) -> pd.DataFrame:
    """Runs recursive_backtest_sku for every Product/Channel combination,
    using the POOLED model of that Channel (same model shared across all
    5 products)."""
    feature_cols = feature_cols or FEATURE_COLS
    all_results = []
    for channel, model in model_dict.items():
        products = raw_df.loc[raw_df["Channel"] == channel, "Product"].unique()
        for product in products:
            r = recursive_backtest_sku(model, raw_df, product, channel, feature_cols=feature_cols)
            if len(r):
                all_results.append(r)
    return pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()


def recursive_backtest_all_sku_level(sku_models: dict, raw_df: pd.DataFrame, feature_cols: list = None) -> pd.DataFrame:
    """
    Same idea as recursive_backtest_all(), but for SKU-LEVEL models (keyed
    by (product, channel), one dedicated model per SKU-channel instead of
    a shared pooled one). This is what lets us compare pooled vs SKU-level
    under the SAME evaluation methodology (recursive, multi-step) instead
    of mixing one-step-ahead for one and recursive for the other.
    """
    feature_cols = feature_cols or FEATURE_COLS
    all_results = []
    for (product, channel), model in sku_models.items():
        r = recursive_backtest_sku(model, raw_df, product, channel, feature_cols=feature_cols)
        if len(r):
            all_results.append(r)
    return pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()


def degradation_curve(recursive_results: pd.DataFrame) -> pd.DataFrame:
    """Aggregates recursive backtest results into Accuracy (1-MAPE) by horizon."""
    return (
        recursive_results.groupby("Horizon")
        .agg(Accuracy=("APE", lambda x: 1 - x.mean()), n=("APE", "size"))
        .reset_index()
    )


def recursive_accuracy_by_sku(recursive_results: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregates a recursive backtest's row-level results (Product, Channel,
    Horizon, APE) into a single Accuracy per Product x Channel, across ALL
    horizons combined -- i.e. "how good is this model, on average, across
    the full 23-week recursive forecast for this SKU-channel?"
    """
    return (
        recursive_results.groupby(["Product", "Channel"])
        .agg(Accuracy=("APE", lambda x: 1 - x.mean()), n=("APE", "size"))
        .reset_index()
    )


def plot_degradation_curve(curve_baseline: pd.DataFrame, curve_improved: pd.DataFrame, out_path: str):
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.plot(curve_baseline["Horizon"], curve_baseline["Accuracy"] * 100, marker="o", label="Baseline", color="#d62728")
    ax.plot(curve_improved["Horizon"], curve_improved["Accuracy"] * 100, marker="o", label="Improved", color="#2ca02c")
    ax.set_xlabel("Horizonte (semanas à frente, sem dado real intermediário)")
    ax.set_ylabel("Accuracy (1 - MAPE) %")
    ax.set_title("Degradação da acurácia por horizonte\n(previsão recursiva / multi-step, todos os SKU-canal)")
    ax.axhline(75, color="gray", linestyle="--", linewidth=1, label="Benchmark (75%)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_routing_recommendation(routing: pd.DataFrame, out_path: str):
    """Bar chart of Accuracy_delta_pp (SKU-level - Pooled) per SKU-channel, colored by recommendation."""
    df = routing.copy()
    df["Label"] = df["Product"].str.replace(" 1L| 350ml| 500ml| 200g| 50g", "", regex=True) + " / " + df["Channel"]
    color_map = {
        "SKU-level": "#2ca02c",
        "Pooled": "#1f77b4",
        "Pooled (SKU-level unstable)": "#d62728",
    }
    colors = df["Recommendation"].map(color_map)

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(df["Label"], df["Accuracy_delta_pp"], color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Accuracy_delta_pp (SKU-level − Pooled, pontos percentuais)")
    ax.set_title("Roteamento por SKU-canal: Pooled vs SKU-level")

    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, label=l) for l, c in color_map.items()]
    ax.legend(handles=legend_elements, loc="lower right")

    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_bakeoff(bakeoff: pd.DataFrame, model_labels: list, out_path: str):
    """Grouped bar chart: one group per SKU-channel, one bar per candidate model."""
    df = bakeoff.copy()
    df["Label"] = df["Product"].str.replace(" 1L| 350ml| 500ml| 200g| 50g", "", regex=True) + " / " + df["Channel"]

    fig, ax = plt.subplots(figsize=(10, 7))
    y = np.arange(len(df))
    n_models = len(model_labels)
    height = 0.8 / n_models
    colors = plt.cm.tab10(np.linspace(0, 1, n_models))

    for i, label in enumerate(model_labels):
        offset = (i - (n_models - 1) / 2) * height
        ax.barh(y + offset, df[label] * 100, height=height, label=label, color=colors[i])

    ax.set_yticks(y)
    ax.set_yticklabels(df["Label"])
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlabel("Accuracy (1 - MAPE) %, avaliação recursiva")
    ax.set_title("Bake-off de modelos por SKU-canal\n(LightGBM Pooled/SKU-level vs XGBoost SKU-level vs GAM SKU-level)")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


# ---------------------------------------------------------------------------
# 8. PLOTS
# ---------------------------------------------------------------------------

def plot_accuracy_comparison(comparison: pd.DataFrame, out_path: str):
    plot_df = comparison[comparison["Channel"] != "OVERALL (weighted)"]
    fig, ax = plt.subplots(figsize=(7, 4.5))
    x = np.arange(len(plot_df))
    width = 0.35
    ax.bar(x - width / 2, plot_df["Accuracy_baseline"] * 100, width, label="Baseline (atual)")
    ax.bar(x + width / 2, plot_df["Accuracy_improved"] * 100, width, label="Improved (regularizado + monotonic)")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["Channel"])
    ax.set_ylabel("Accuracy (1 - MAPE) %")
    ax.set_title("Accuracy by channel — Baseline vs Improved")
    ax.axhline(75, color="gray", linestyle="--", linewidth=1, label="Industry benchmark (75%)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def plot_price_sign(sign_baseline: dict, sign_improved: dict, out_path: str):
    variants = ["Baseline", "Improved"]
    values = [sign_baseline["pct_positive_sign"], sign_improved["pct_positive_sign"]]
    fig, ax = plt.subplots(figsize=(5.5, 4))
    bars = ax.bar(variants, values, color=["#d62728", "#2ca02c"])
    ax.set_ylabel("% of weeks with WRONG-signed price SHAP (positive)")
    ax.set_title("Price elasticity sign consistency\nNatural Juice / Supermarkets")
    ax.set_ylim(0, 100)
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width() / 2, v + 2, f"{v:.0f}%", ha="center")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


# ---------------------------------------------------------------------------
# 9. MAIN
# ---------------------------------------------------------------------------

def main():
    outdir = Path(OUTDIR)
    outdir.mkdir(exist_ok=True, parents=True)

    # --- Load ---
    print("1) Loading raw data (Table 1 + Table 2)...")
    raw = load_data(DATA_PATH)
    print(f"   -> {len(raw)} rows loaded")

    # --- Feature engineering ---
    print("\n2) Engineering features (Table 4 rules)...")
    df = engineer_features(raw)
    print(f"   -> {len(df)} rows after feature engineering / dropna")

    # --- Split ---
    train, test = time_split(df)
    print(f"\n3) Time-based split: {len(train)} train rows / {len(test)} test rows")

    # --- Train ---
    print("\n4) Training BASELINE models (one per channel, current params)...")
    baseline_models = train_all_channels(train, BASELINE_PARAMS)

    print("5) Training IMPROVED models (one per channel, regularized + monotonic)...")
    improved_models = train_all_channels(train, IMPROVED_PARAMS)

    print("5b) Training IMPROVED_V2 models (also constrains Interaction_Price_Promo)...")
    improved_v2_models = train_all_channels(train, IMPROVED_V2_PARAMS)

    print("5c) Training IMPROVED_V3 models (SKU-relative Price_Index instead of raw price)...")
    train_norm, test_norm = add_price_index(train, test)
    improved_v3_models = train_all_channels(train_norm, IMPROVED_V3_PARAMS, feature_cols=FEATURE_COLS_NORM)

    print("5d) Training BASELINE_V3 models (Price_Index, but WITHOUT the monotonic constraint)...")
    # Same hyperparameters as BASELINE_PARAMS (no regularization, no constraint),
    # just fed the normalized price. Isolates: does normalization ALONE fix the
    # SHAP sign issue, or is the monotonic constraint also needed?
    baseline_v3_models = train_all_channels(train_norm, BASELINE_PARAMS, feature_cols=FEATURE_COLS_NORM)

    print("5e) Training SKU_LEVEL models (one per Product x Channel, no pooling)...")
    sku_level_models = train_all_sku_channels(train, SKU_LEVEL_PARAMS)

    # --- Evaluate: Accuracy = 1 - MAPE ---
    print("\n6) Evaluating BASELINE (Accuracy = 1 - MAPE)...")
    baseline_eval = evaluate_all_channels(baseline_models, test)
    print(baseline_eval.to_string(index=False))

    print("\n7) Evaluating IMPROVED (Accuracy = 1 - MAPE)...")
    improved_eval = evaluate_all_channels(improved_models, test)
    print(improved_eval.to_string(index=False))

    print("\n7b) Evaluating IMPROVED_V3 / Price_Index (Accuracy = 1 - MAPE)...")
    improved_v3_eval = evaluate_all_channels(improved_v3_models, test_norm, feature_cols=FEATURE_COLS_NORM)
    print(improved_v3_eval.to_string(index=False))

    print("\n7c) Evaluating BASELINE_V3 / Price_Index, no constraint (Accuracy = 1 - MAPE)...")
    baseline_v3_eval = evaluate_all_channels(baseline_v3_models, test_norm, feature_cols=FEATURE_COLS_NORM)
    print(baseline_v3_eval.to_string(index=False))

    print("\n7d) Evaluating SKU_LEVEL models (one per Product x Channel) -- per SKU-channel:")
    sku_level_eval = evaluate_all_sku_channels(sku_level_models, test)
    print(sku_level_eval.to_string(index=False))

    print("\n7e) SKU_LEVEL rolled up to channel level (apples-to-apples vs pooled models):")
    sku_level_by_channel = aggregate_sku_eval_by_channel(sku_level_eval)
    print(sku_level_by_channel.to_string(index=False))
    print("   (Compare this Accuracy column directly against Baseline/Improved in step 6/7 --")
    print("    same test weeks, same weighting, only difference is pooling vs no pooling.)")

    print("\n7f) [PRELIMINARY] Routing recommendation: Pooled vs SKU-level, one-step-ahead...")
    print("    (Uses real Sell_In for lags -- NOT representative of the 78-week forecast.")
    print("    See step 11c for the recursive version, which should drive the final decision.)")
    pooled_by_sku = evaluate_pooled_by_sku(improved_models, test)
    routing = compare_pooled_vs_sku_level(pooled_by_sku, sku_level_eval, min_accuracy=0.0)
    print(routing.to_string(index=False))
    print(f"\n   -> {(routing['Recommendation']=='SKU-level').sum()} de {len(routing)} combinações recomendadas para SKU-level (preliminar)")
    routing.to_csv(outdir / "routing_recommendation_onestep_PRELIMINARY.csv", index=False)

    comparison = compare_baseline_vs_improved(baseline_eval, improved_eval)
    print("\n8) Baseline vs Improved comparison:")
    print(comparison.to_string(index=False))
    comparison.to_csv(outdir / "accuracy_comparison.csv", index=False)

    # --- SHAP validation ---
    print("\n9) Validating Problem 2 fix — price elasticity sign (Natural Juice / Supermarkets)...")
    full_df = pd.concat([train, test])
    sign_baseline = price_sign_consistency(baseline_models, full_df)
    sign_improved = price_sign_consistency(improved_models, full_df)
    sign_improved_v2 = price_sign_consistency(improved_v2_models, full_df)
    print(f"   Baseline:      {sign_baseline['pct_positive_sign']:.1f}% wrong-signed weeks, mean SHAP={sign_baseline['mean_shap']:.4f}")
    print(f"   Improved:      {sign_improved['pct_positive_sign']:.1f}% wrong-signed weeks, mean SHAP={sign_improved['mean_shap']:.4f}")
    print(f"   Improved_V2:   {sign_improved_v2['pct_positive_sign']:.1f}% wrong-signed weeks, mean SHAP={sign_improved_v2['mean_shap']:.4f}")
    print("   (V2 also constrains Interaction_Price_Promo -- tests the 'leakage through")
    print("    the interaction term' hypothesis. If V2's % drops a lot vs Improved, the")
    print("    hypothesis is confirmed.)")

    full_df_norm = pd.concat([train_norm, test_norm])
    sign_baseline_v3 = price_sign_consistency(
        baseline_v3_models, full_df_norm, feature_cols=FEATURE_COLS_NORM, price_col="Price_Index"
    )
    sign_improved_v3 = price_sign_consistency(
        improved_v3_models, full_df_norm, feature_cols=FEATURE_COLS_NORM, price_col="Price_Index"
    )
    print(f"   Baseline_V3 (normalization ONLY, no constraint): {sign_baseline_v3['pct_positive_sign']:.1f}% wrong-signed, mean SHAP={sign_baseline_v3['mean_shap']:.4f}")
    print(f"   Improved_V3 (normalization + constraint):        {sign_improved_v3['pct_positive_sign']:.1f}% wrong-signed, mean SHAP={sign_improved_v3['mean_shap']:.4f}")
    print("   (Comparing these two isolates whether normalization alone is enough, or")
    print("    whether the monotonic constraint is still doing real work on top of it.)")

    sku_model_nj_sup = sku_level_models.get(("Natural Juice 1L", "Supermarkets"))
    sign_sku_level = None
    if sku_model_nj_sup is not None:
        sign_sku_level = price_sign_consistency(
            df=full_df, product="Natural Juice 1L", channel="Supermarkets", model=sku_model_nj_sup
        )
        print(f"   SKU_LEVEL (no pooling, raw price, no constraint): {sign_sku_level['pct_positive_sign']:.1f}% wrong-signed, mean SHAP={sign_sku_level['mean_shap']:.4f}")
        print("   (Bonus check: does training per-SKU alone -- with no normalization and no")
        print("    monotonic constraint -- already fix the sign issue, since the model never")
        print("    sees other products' price scales in the first place?)")

    print("\n10) Validating Promotion vs Advertising attribution (E-commerce / Beverages)...")
    share_baseline = promo_vs_advertising_share(baseline_models, full_df)
    share_improved = promo_vs_advertising_share(improved_models, full_df)
    print(f"   Baseline: Promotion={share_baseline['promo_share_pct']:.1f}%  Advertising={share_baseline['advertising_share_pct']:.1f}%")
    print(f"   Improved: Promotion={share_improved['promo_share_pct']:.1f}%  Advertising={share_improved['advertising_share_pct']:.1f}%")

    print("\n11) Computing partial dependence (direct proof of the monotonic constraint)...")
    pdp_baseline = price_partial_dependence(baseline_models["Supermarkets"], full_df)
    pdp_improved = price_partial_dependence(improved_models["Supermarkets"], full_df)
    print("   Baseline PDP (should NOT be monotonic decreasing):")
    print(pdp_baseline.to_string(index=False))
    print("   Improved PDP (MUST be monotonic non-increasing):")
    print(pdp_improved.to_string(index=False))

    pdp_baseline_v3 = price_partial_dependence(
        baseline_v3_models["Supermarkets"], full_df_norm, feature_cols=FEATURE_COLS_NORM, price_col="Price_Index"
    )
    pdp_improved_v3 = price_partial_dependence(
        improved_v3_models["Supermarkets"], full_df_norm, feature_cols=FEATURE_COLS_NORM, price_col="Price_Index"
    )

    print("\n11b) Running RECURSIVE (multi-step) backtest -- simulates the 78-week")
    print("     future forecast, where lags come from predictions, not real data...")
    recursive_baseline = recursive_backtest_all(baseline_models, raw)
    recursive_improved = recursive_backtest_all(improved_models, raw)
    curve_baseline = degradation_curve(recursive_baseline)
    curve_improved = degradation_curve(recursive_improved)
    print("   Accuracy by horizon -- Baseline:")
    print(curve_baseline.to_string(index=False))
    print("   Accuracy by horizon -- Improved:")
    print(curve_improved.to_string(index=False))
    recursive_baseline.to_csv(outdir / "recursive_backtest_baseline.csv", index=False)
    recursive_improved.to_csv(outdir / "recursive_backtest_improved.csv", index=False)

    print("\n11c) [FINAL] Routing recommendation: Pooled vs SKU-level, RECURSIVE (multi-step)...")
    print("     Both sides now evaluated the same way -- lags built from predictions, not")
    print("     real data -- matching the actual 78-week forecast scenario. This table")
    print("     supersedes the preliminary one from step 7f.")
    recursive_sku_level = recursive_backtest_all_sku_level(sku_level_models, raw)
    recursive_sku_level.to_csv(outdir / "recursive_backtest_sku_level.csv", index=False)

    routing_recursive = compare_pooled_vs_sku_level_recursive(
        recursive_improved, recursive_sku_level, min_accuracy=0.0
    )
    print(routing_recursive.to_string(index=False))
    print(f"\n   -> {(routing_recursive['Recommendation']=='SKU-level').sum()} de {len(routing_recursive)} combinações recomendadas para SKU-level (final, recursivo)")
    routing_recursive.to_csv(outdir / "routing_recommendation_FINAL.csv", index=False)

    # Sanity check: did the recursive evaluation flip any recommendation vs the
    # preliminary one-step-ahead table? Worth flagging explicitly either way.
    flip_check = routing[["Product", "Channel", "Recommendation"]].merge(
        routing_recursive[["Product", "Channel", "Recommendation"]],
        on=["Product", "Channel"], suffixes=("_onestep", "_recursive"),
    )
    n_flipped = (flip_check["Recommendation_onestep"] != flip_check["Recommendation_recursive"]).sum()
    print(f"\n   -> {n_flipped} de {len(flip_check)} recomendações MUDARAM entre a avaliação one-step e a recursiva.")
    if n_flipped > 0:
        print(flip_check[flip_check["Recommendation_onestep"] != flip_check["Recommendation_recursive"]].to_string(index=False))

    # --- Model bake-off: LightGBM (pooled + SKU-level) vs XGBoost (SKU-level)
    # vs GAM (SKU-level), all compared under the RECURSIVE methodology ---
    print("\n13) Training XGBoost SKU-level models...")
    xgb_sku_models = train_all_sku_channels(train, XGB_SKU_PARAMS, feature_cols=FEATURE_COLS, model_class=xgb.XGBRegressor)

    print("13b) Training GAM SKU-level models (explicit trend + seasonality terms)...")
    gam_sku_models = train_gam_sku_channels(train, feature_cols=FEATURE_COLS_GAM)

    print("13c) Running recursive backtest for XGBoost and GAM...")
    recursive_xgb_sku = recursive_backtest_all_sku_level(xgb_sku_models, raw, feature_cols=FEATURE_COLS)
    recursive_gam_sku = recursive_backtest_all_sku_level(gam_sku_models, raw, feature_cols=FEATURE_COLS_GAM)
    recursive_xgb_sku.to_csv(outdir / "recursive_backtest_xgboost_sku_level.csv", index=False)
    recursive_gam_sku.to_csv(outdir / "recursive_backtest_gam_sku_level.csv", index=False)

    print("\n13d) [BAKE-OFF] Comparing all 4 candidates per SKU-channel (recursive accuracy)...")
    bakeoff = build_model_bakeoff_table(
        {
            "Pooled_LightGBM": recursive_accuracy_by_sku(recursive_improved),
            "SKU_LightGBM": recursive_accuracy_by_sku(recursive_sku_level),
            "SKU_XGBoost": recursive_accuracy_by_sku(recursive_xgb_sku),
            "SKU_GAM": recursive_accuracy_by_sku(recursive_gam_sku),
        },
        min_accuracy=0.0,
    )
    print(bakeoff.to_string(index=False))
    print("\n   Win count by model family:")
    print(bakeoff["Recommendation"].value_counts().to_string())
    bakeoff.to_csv(outdir / "model_bakeoff_FINAL.csv", index=False)

    print("\n13e) Bonus checks: does XGBoost/GAM also fix the price sign issue")
    print("     (Natural Juice / Supermarkets), same as SKU-level LightGBM did?")
    xgb_model_nj_sup = xgb_sku_models.get(("Natural Juice 1L", "Supermarkets"))
    if xgb_model_nj_sup is not None:
        sign_xgb = price_sign_consistency(
            df=full_df, product="Natural Juice 1L", channel="Supermarkets",
            model=xgb_model_nj_sup, feature_cols=FEATURE_COLS,
        )
        print(f"   XGBoost SKU-level: {sign_xgb['pct_positive_sign']:.1f}% wrong-signed, mean SHAP={sign_xgb['mean_shap']:.4f}")

    gam_model_nj_sup = gam_sku_models.get(("Natural Juice 1L", "Supermarkets"))
    pdp_gam_price = None
    if gam_model_nj_sup is not None:
        # GAM isn't a tree model -- SHAP's TreeExplainer doesn't apply. Use
        # partial dependence instead (works generically via .predict()),
        # same check we've used throughout for the tree models.
        full_df_gam = pd.concat([train, test])
        pdp_gam_price = price_partial_dependence(
            gam_model_nj_sup, full_df_gam, feature_cols=FEATURE_COLS_GAM, price_col="Price_per_kg_USD"
        )
        print("   GAM SKU-level price partial dependence (should be non-increasing):")
        print(pdp_gam_price.to_string(index=False))

    # --- Plots ---
    print("\n12) Generating plots...")
    plot_accuracy_comparison(comparison, str(outdir / "accuracy_by_channel.png"))
    plot_price_sign(sign_baseline, sign_improved, str(outdir / "price_sign_consistency.png"))
    plot_partial_dependence(pdp_baseline, pdp_improved, str(outdir / "price_partial_dependence.png"))
    plot_partial_dependence(
        pdp_baseline_v3, pdp_improved_v3, str(outdir / "price_index_partial_dependence.png"),
        price_col="Price_Index", x_label="Price_Index (1.0 = SKU's own average price)",
        title_suffix=" (normalized)",
        label_baseline="Normalized only (no constraint)",
        label_improved="Normalized + monotonic",
    )
    plot_degradation_curve(curve_baseline, curve_improved, str(outdir / "accuracy_degradation_by_horizon.png"))
    plot_routing_recommendation(routing, str(outdir / "routing_recommendation_onestep_PRELIMINARY.png"))
    plot_routing_recommendation(routing_recursive, str(outdir / "routing_recommendation_FINAL.png"))
    plot_bakeoff(
        bakeoff, ["Pooled_LightGBM", "SKU_LightGBM", "SKU_XGBoost", "SKU_GAM"],
        str(outdir / "model_bakeoff_FINAL.png"),
    )

    print(f"\nDone. Results saved to: {outdir.resolve()}")

    # Return everything so it can be inspected in a notebook, e.g.:
    #   results = main()
    #   print(results["sign_baseline"])
    return {
        "raw": raw,
        "df": df,
        "train": train,
        "test": test,
        "train_norm": train_norm,
        "test_norm": test_norm,
        "baseline_models": baseline_models,
        "improved_models": improved_models,
        "improved_v2_models": improved_v2_models,
        "improved_v3_models": improved_v3_models,
        "baseline_v3_models": baseline_v3_models,
        "sku_level_models": sku_level_models,
        "baseline_eval": baseline_eval,
        "improved_eval": improved_eval,
        "improved_v3_eval": improved_v3_eval,
        "baseline_v3_eval": baseline_v3_eval,
        "sku_level_eval": sku_level_eval,
        "sku_level_by_channel": sku_level_by_channel,
        "pooled_by_sku": pooled_by_sku,
        "routing": routing,
        "recursive_sku_level": recursive_sku_level,
        "routing_recursive": routing_recursive,
        "xgb_sku_models": xgb_sku_models,
        "gam_sku_models": gam_sku_models,
        "recursive_xgb_sku": recursive_xgb_sku,
        "recursive_gam_sku": recursive_gam_sku,
        "bakeoff": bakeoff,
        "pdp_gam_price": pdp_gam_price,
        "comparison": comparison,
        "sign_baseline": sign_baseline,
        "sign_improved": sign_improved,
        "sign_improved_v2": sign_improved_v2,
        "sign_improved_v3": sign_improved_v3,
        "sign_baseline_v3": sign_baseline_v3,
        "sign_sku_level": sign_sku_level,
        "share_baseline": share_baseline,
        "share_improved": share_improved,
        "pdp_baseline": pdp_baseline,
        "pdp_improved": pdp_improved,
        "pdp_improved_v3": pdp_improved_v3,
        "recursive_baseline": recursive_baseline,
        "recursive_improved": recursive_improved,
        "curve_baseline": curve_baseline,
        "curve_improved": curve_improved,
    }


if __name__ == "__main__":
    main()

1) Loading raw data (Table 1 + Table 2)...
   -> 2340 rows loaded

2) Engineering features (Table 4 rules)...
   -> 2280 rows after feature engineering / dropna

3) Time-based split: 1935 train rows / 345 test rows

4) Training BASELINE models (one per channel, current params)...
5) Training IMPROVED models (one per channel, regularized + monotonic)...
5b) Training IMPROVED_V2 models (also constrains Interaction_Price_Promo)...
5c) Training IMPROVED_V3 models (SKU-relative Price_Index instead of raw price)...
5d) Training BASELINE_V3 models (Price_Index, but WITHOUT the monotonic constraint)...
